# 02 - Static wind loads

For each wind direction: `Cp = (p - p_ref)/q`, per-floor force/moment
coefficients (`cfdmod.building.per_floor_loads`), then dimensional
static-equivalent floor loads. Produces the per-direction floor Fx/Fy/Mz
profiles, the directional global envelope, and the Eberick peak-load tables.

Same shape as the old `hfpi_static.ipynb`: config read inline, the
direction loop is right here in a cell, heavy compute called straight from
`cfdmod`. Run it against any kernel that has cfdmod v3.

In [ ]:
import json, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
%matplotlib inline
from matplotlib.ticker import FuncFormatter

from cfdmod import plot_config
from cfdmod.adapters.xdmf_h5 import XdmfH5Storage
from cfdmod.adapters.memory import MemoryFieldStore
from cfdmod.core import PointsDataSource, Topology, ElementMeta
from cfdmod.core.container import Container
from cfdmod.building import BuildingCase, per_floor_loads
from cfdmod.analytical import WindProfile_NBR
from cfdmod.dynamics import (
    DimensionalData, BuildingCaseParameters,
    get_stats_forces_effective, filter_by_xi, join_by_direction,
)
from cfdmod.dynamics.plotting import (
    plot_floor_by_floor_mean_peaks, plot_global_stats_per_direction,
    effective_peak_loads_per_direction, add_eberick_floor_height_and_pavements,
)

import logging
logging.getLogger("cfdmod").setLevel(logging.WARNING)  # quiet per-direction INFO spam
plot_config.apply_style()
TF = 1.0 / 9806.65  # N -> tonne-force

## Case config (read inline from the existing case_data)

In [ ]:
project = pathlib.Path("/data/eng/consulting/NNN_CaseName")
case_data = project / "post_processing/pp_config/case_data"
results   = project / "results"

dct_case = json.loads((case_data / "global_data.json").read_text())
dct_case

In [ ]:
from ruamel.yaml import YAML
yaml = YAML(typ="safe")

batch = dct_case["analysis"]["batch_name"]
H  = float(dct_case["H"])
L  = float(dct_case["L"])
V0 = float(dct_case["V0"])
bodies = dct_case["analysis"]["body_names"]

# terrain categories -> directions (each category has its own inlet U_H)
cats = sorted(k.split("directions_cat")[1] for k in dct_case["analysis"]            if k.startswith("directions_cat"))
dirs_by_cat = {c: [str(d) for d in dct_case["analysis"][f"directions_cat{c}"]] for c in cats}

# per-category simulation U_H + reference area/volume + per-body floor z-edges,
# read straight from the params_cat*.yaml the case already ships
simul_u_h, floor_heights = {}, {}
for c in cats:
    params = yaml.load((case_data / f"params_cat{c}.yaml").open())
    cp0 = next(iter(params["pressure_coefficient"].values()))
    simul_u_h[c] = float(cp0["simul_U_H"])
    fluid_density = float(cp0.get("fluid_density", 1.225))
    fc0 = next(iter(params["force_coefficient"].values()))
    nominal_area   = float(fc0["nominal_area"])
    nominal_volume = float(next(iter(params["moment_coefficient"].values()))["nominal_volume"])
    if not floor_heights:
        for b in bodies:
            blk = params["force_coefficient"].get(f"building_{b}") or fc0
            floor_heights[b] = [float(z) for z in blk["bodies"][0]["sub_bodies"]["z_intervals"]]

directions = [d for c in cats for d in dirs_by_cat[c]]
category_of = lambda d: next(c for c, ds in dirs_by_cat.items() if d in ds)
print(f"H={H} L={L} V0={V0} | bodies={bodies}")
print(f"cats={cats} simul_U_H={simul_u_h} | {len(directions)} directions")

# design speeds computed in 01_inflow are handed over via a local _shared.json,
# so we do not copy-paste them between notebooks
shared = {}
sp = pathlib.Path.cwd() / "_shared.json"
if sp.exists():
    shared = json.loads(sp.read_text())
    print("loaded _shared.json from 01_inflow:", list(shared))
else:
    print("no _shared.json (run 01_inflow first); design speed computed inline instead")
design_by_dir = {float(k): v for k, v in shared.get("design_U_H_by_direction_NBR", {}).items()}

## Pick a body + tiny glue (kept visible here)

Two small helpers cfdmod v3 does not yet expose as one call -- slated to move
into `cfdmod.building` so these cells become a single import. For now they are
right here so you can read/change them:
- `read_point_probe`: the reference-pressure probe has no `/Triangles`, so
  `XdmfH5Storage` rejects it; read it straight into a `PointsDataSource`.
- `static_response`: dimensionalize per-floor Cf/Cm into static-equivalent
  floor loads (`DimensionalData` convention: `F = cf * base * H * q`), keeping
  empty floor bands so the profile stays aligned to the storey table.

In [ ]:
body = "base"
# body = "torre01"
# body = "torre02"
# body = "torre03"
# body = "torre04"

mesh_path = project / f"run/artifacts/lnas/{body}.lnas"
probe_folder = lambda d: results / batch / f"BaciadaVovo_{d}" / "000/probes/hist_series" / f"cp_analysis_{body}"

In [ ]:
def read_point_probe(h5_path, time_axis):
    """Reference-pressure probe (points.point0.h5) -> PointsDataSource."""
    with h5py.File(h5_path, "r") as h:
        grp = h["pressure"]
        keys = sorted(grp.keys(), key=lambda k: float(k[1:]))
        arr = np.empty((1, len(keys)), dtype=np.float32)
        for i, k in enumerate(keys):
            arr[0, i] = grp[k][()][0]
        pts = np.zeros((1, 3))
        if "Geometry" in h:
            pts[0] = np.asarray(h["Geometry"])[0]
    return PointsDataSource(time=time_axis, topology=Topology.points(pts),
                            elements=ElementMeta(position=pts),
                            fields=MemoryFieldStore({"pressure": arr}))


def static_response(cf, cm, case, load_u_h):
    """Per-floor Cf/Cm -> static-equivalent floor loads (feq_x/feq_y/meq_z)."""
    edges = np.asarray(case.floor_heights)
    n_bands = len(edges) - 1
    # zoning drops empty z-bands; recover which bands hold geometry to keep alignment
    from lnas import LnasFormat
    geo = LnasFormat.from_file(mesh_path).geometry
    cz = np.asarray(geo.vertices)[np.asarray(geo.triangles)][:, :, 2].mean(axis=1)
    bands = np.where(np.histogram(cz, bins=edges)[0] > 0)[0]

    def scatter(rows):
        rows = np.asarray(rows, dtype=np.float64)
        if rows.shape[0] == n_bands:
            return rows
        full = np.zeros((n_bands, rows.shape[1]))
        full[bands[-rows.shape[0]:]] = rows
        return full

    dim = DimensionalData(U_H=load_u_h, height=case.reference_height,
                          base=case.characteristic_length, integral_scale_multiplier=1.0)
    fx = scatter(cf.fields.read("cf_x")) * dim.force_normalization_factor
    fy = scatter(cf.fields.read("cf_y")) * dim.force_normalization_factor
    mz = scatter(cm.fields.read("cm_z")) * dim.moments_normalization_factor
    z_mid = 0.5 * (edges[:-1] + edges[1:])
    pts = np.zeros((n_bands, 3)); pts[:, 2] = z_mid
    return PointsDataSource(time=cf.time, topology=Topology.points(pts),
                            elements=ElementMeta(position=pts),
                            fields=MemoryFieldStore({"feq_x": fx, "feq_y": fy, "meq_z": mz}))

## Solve every direction (the loop, right here)

One direction at a time so the multi-GB body pressure array is released before
the next (`chunk_size` streams the Cp -> Cf/Cm chain). Cp is non-dimensionalized
by the simulation inlet U_H (in the `BuildingCase`); the static loads here are
referenced at that same `simul` U_H (swap to the NBR design U_H via the
commented `wp.get_U_H(...)` line for full-design forces).

In [ ]:
wp = WindProfile_NBR.build(case_data / "wind_analysis_NBR.csv", V0=V0)

responses = {}
for d in directions:
    cat = category_of(d)
    folder = probe_folder(d)
    if not (folder / f"bodies.{body}.h5").exists():
        print(f"  [skip] {body} {d}: no data"); continue
    storage = XdmfH5Storage(folder)
    bdy  = storage.read_data_source(pathlib.Path(f"bodies.{body}"))
    pref = read_point_probe(folder / "points.point0.h5", bdy.time)
    case = BuildingCase(name=body, reference_height=H, characteristic_length=L,
                        basic_wind_speed=V0, simul_reference_velocity=simul_u_h[cat],
                        nominal_area=nominal_area, nominal_volume=nominal_volume,
                        floor_heights=floor_heights[body], fluid_density=fluid_density)
    cf, cm = per_floor_loads(bdy, pref, str(mesh_path), case, method="centroid", chunk_size=3000)
    # design speed: from 01_inflow (_shared.json) if present, else computed inline
    design_u_h = design_by_dir.get(float(d)) or wp.get_U_H(H, float(d), 50, time_filter_seconds=3, use_kd=True)
    load_u_h = simul_u_h[cat]        # default: reference at the simulation inlet speed
    # load_u_h = design_u_h          # alternative: 50-year NBR design speed
    responses[float(d)] = static_response(cf, cm, case, load_u_h)
    del bdy, pref
    print(f"  {body} {d} (cat{cat}): load U_H={load_u_h:.2f}")

sorted(responses)

## Directional global envelope (base loads per direction)

In [ ]:
# pack into the dynamics Container shape so the stock reducers/plotters work
cont = Container(items={
    BuildingCaseParameters(direction=d, xi=0.0, recurrence_period=50.0, use_kd=True): r
    for d, r in responses.items()
})

# global base loads (sum floors) + overturning Mx/My (force x lever arm) + torsion Mz
rows_f, rows_m = [], []
for d in sorted(responses):
    r = responses[d]
    z  = np.asarray(r.elements.position)[:, 2]
    fx = np.asarray(r.fields.read("feq_x")); fy = np.asarray(r.fields.read("feq_y"))
    mz = np.asarray(r.fields.read("meq_z"))
    g = {"Fx": fx.sum(0), "Fy": fy.sum(0), "Mx": (fy * z[:, None]).sum(0),
         "My": (fx * z[:, None]).sum(0), "Mz": mz.sum(0)}
    rf = {"direction": d}; rm = {"direction": d}
    for s, red in (("min", np.min), ("max", np.max), ("mean", np.mean)):
        rf[f"{s}_x"] = red(g["Fx"]); rf[f"{s}_y"] = red(g["Fy"])
        rm[f"{s}_x"] = red(g["Mx"]); rm[f"{s}_y"] = red(g["My"]); rm[f"{s}_z"] = red(g["Mz"])
    rows_f.append(rf); rows_m.append(rm)
stats = {"forces_static": pd.DataFrame(rows_f), "moments_static": pd.DataFrame(rows_m)}

fig, _ = plot_global_stats_per_direction({0.0: stats}, unit_conversion=TF,
                                          unit_name="tf", variable_types=["static"], xticks=45)
# workaround: cfdmod labels each x-tick with ' deg' (they overlap); use a degree symbol
for ax in fig.axes:
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: rf"${v:.0f}^\circ$"))
plt.show()
stats["forces_static"].round(2)

## Per-direction floor-by-floor Fx / Fy / Mz

In [ ]:
n_floors = next(iter(responses.values())).n_elements
for d in sorted(responses):
    r = responses[d]
    vals = {s: get_stats_forces_effective(r, s) for s in ("min", "max", "mean")}
    fig, _ = plot_floor_by_floor_mean_peaks(vals_plot=vals, vals_labels=("Fx", "Fy", "Mz"),
                                            wind_dir=d, unit_conversion=TF, unit_name="tf",
                                            y_abs=(0, n_floors + 1))
    fig.suptitle(rf"wind ${d:.0f}^\circ$", fontweight="bold")  # workaround: cfdmod title says " deg"
    plt.show()

## Eberick per-direction peak-load tables

In [ ]:
alt = pd.read_csv(case_data / f"alturas_{body}.csv")
tables = effective_peak_loads_per_direction(filter_by_xi(cont, 0.0))
for axis, df in tables.items():
    if len(df) == len(alt):
        add_eberick_floor_height_and_pavements(df, alt["z_min"].to_numpy(), alt["Pavimento"].tolist())
tables["Fx"].round(2)